# B5 · Statistics with AI

*Path B: AI-Assisted, notebook 5 of 5 · about 30 to 40 minutes*

**In this notebook you will learn to:**

- Choose a statistical test before asking for code
- Read the output of a t-test, ANOVA, chi-square test, and regression
- Get odds ratios from a logistic regression, and interpret them
- Carry this way of working over to real *All of Us* data

**Before this notebook:** [B4: Catching Mistakes](https://colab.research.google.com/github/icohr-lab/tutorials/blob/main/notebooks/B4_Catching_Mistakes.ipynb). This notebook assumes you have met these tests in a statistics course.

> **Before you start:** click **File > Save a copy in Drive** and work in your copy. Run a code cell by clicking the play button to its left, or press **Shift + Enter**. Go in order from top to bottom.

## 1. What the AI will not do for you

An AI assistant will write code for any test you ask for, on any data, whether or not the test is appropriate. It will also produce a confident interpretation of whatever comes out. **Choosing the test and judging what the result means are your job.**

Run the setup cell. The lines printed by `library(tidyverse)` and `read_csv()` are normal. It also loads `broom`, a package that turns model results into tidy tables.

In [ ]:
library(tidyverse)
library(broom)

study_df <- read_csv("https://raw.githubusercontent.com/icohr-lab/aou-workshop/refs/heads/main/data/final_dataset2.csv")

# ---- Helper code. You do not need to read this part; just run the cell. ----
describe_columns <- function(data) {
  tibble(column = names(data), type = map_chr(data, ~ class(.x)[1]))
}
check <- function(ok, label) {
  ok <- tryCatch(suppressWarnings(isTRUE(ok)), error = function(e) FALSE)
  cat(if (ok) "PASS    " else "NOT YET ", label, "\n")
}

## 2. Which test?

| Outcome | Compared across, or related to | Method |
|---|---|---|
| A number (e.g. blood pressure) | 2 groups | Two-sample t-test |
| A number | 3 or more groups | One-way ANOVA |
| A category (e.g. disease yes/no) | Groups | Chi-square test |
| A number | Another number | Linear regression |
| Yes/no | One or more predictors | Logistic regression |

The last two rows can overlap with the others. When you simply want to know whether two yes/no variables are associated, with no adjustment for anything else, use the chi-square test.

## 3. A worked example: the t-test

**Question:** is mean systolic blood pressure different in people with and without the condition?

**Choose the test.** The outcome, `SBP`, is a number. It is compared across two groups (`condition_status` is 0 or 1). That calls for a two-sample t-test.

**The prompt:**

> In R, I have a dataframe `study_df` with one row per person. `SBP` is numeric, with a few missing values. `condition_status` is numeric, 0 or 1. Run a two-sample t-test comparing mean SBP between the two condition_status groups.

**The code an AI assistant returned:**

In [ ]:
t.test(SBP ~ condition_status, data = study_df)

**Read the code.** `SBP ~ condition_status` is a **formula**: the outcome, a tilde `~`, and the grouping variable. Read it as "SBP by condition status".

**Read the output.**

- **`p-value`**: 0.84. No evidence of a difference in mean blood pressure between the groups.
- **`95 percent confidence interval`**: the plausible range for the difference between the two means, about -5 to 6. It includes 0.
- **`sample estimates`**: the two group means, 117.4 and 116.8.
- **The title**: "Welch Two Sample t-test".

**Ask about what you did not choose.** You asked for "a two-sample t-test"; the output says "Welch". A good follow-up: *"Which version of the t-test is this, and why is it the default?"* (It is the version that does not assume the two groups have equal variance, which is generally the safer choice.) Whenever the output names something you did not ask for, find out what it is.

## 4. Your turn: three more questions

For each question, **decide which method to use before you ask the AI**, using the table in section 2. Write your choice down. Then prompt for the code, run it, and compare with the expected results underneath.

| | Question | My choice of method |
|---|---|---|
| a | Does mean `SBP` differ among the three `sex_at_birth` groups? | |
| b | Is drug exposure (`exposure_status`, 0 or 1) associated with the condition (`condition_status`, 0 or 1)? | |
| c | How does `SBP` change with `age`? | |

**My prompt:** *(double-click this cell and paste the prompt you used)*

In [ ]:
# (a) Paste the AI's code here, read it, then run it

In [ ]:
# (b) Paste the AI's code here, read it, then run it

In [ ]:
# (c) Paste the AI's code here, read it, then run it

### Expected results

| | Method | Result |
|---|---|---|
| a | One-way ANOVA | p = 0.66 |
| b | Chi-square test | p = 4.76e-06 |
| c | Linear regression | slope for age = 0.51, p < 0.001 |

- `4.76e-06` is scientific notation: move the decimal point 6 places to the left, giving 0.00000476. A very small p-value.
- The slope of 0.51 means blood pressure is higher by about half a mmHg for each additional year of age.

**If your numbers differ, find out why before moving on.** A common reason is that the AI chose a different variant of the test. For (b), for example, R's chi-square test applies a "continuity correction" by default; code that switches it off gives p = 1.56e-06, and Fisher's exact test gives p = 2.9e-06. All three lead to the same conclusion here, but you should know which one you ran. Ask the AI which variant it used and why.

## 5. Logistic regression and odds ratios

**The research question:** is drug exposure associated with the condition, after adjusting for age and sex?

The outcome, `condition_status`, is yes or no, so the method is **logistic regression**. Three things to know in order to read its output:

- The **odds** of an outcome are the probability that it happens divided by the probability that it does not.
- Logistic regression models the **logarithm of the odds** of the outcome.
- So the coefficient it reports for each predictor is a **log odds ratio**.
- To get an **odds ratio**, the logarithm has to be reversed with the exponential function, which in R is `exp()`. This is called exponentiating.

An odds ratio of 1 means no association; above 1 means higher odds of the outcome; below 1 means lower odds.

### The goal

Fit a logistic regression of `condition_status` on `exposure_status`, adjusting for `age` and `sex_at_birth`. Produce a table of **odds ratios with 95% confidence intervals**, and save that table as `tidy_results`.

In your prompt, ask for the table to be made with `tidy()` from the broom package, keeping its standard column names. The provided check looks for those names.

**My prompt:** *(double-click this cell and paste the prompt you used)*

In [ ]:
# Paste the AI's code here, read it, then run it

In [ ]:
# Provided check (you do not need to read this code)
or_row <- tryCatch(tidy_results[grepl("exposure_status", tidy_results$term), ][1, ], error = function(e) NULL)
check(abs(or_row$estimate - 8.64) < 0.06, "the odds ratio for exposure_status is about 8.64")
check(or_row$conf.low > 3.1 && or_row$conf.low < 3.5 && or_row$conf.high > 23 && or_row$conf.high < 25, "its 95% confidence interval runs from a little over 3 to about 24")

If the first line says `NOT YET` and your table shows about 2.16, the coefficients have not been exponentiated. If the table has different column names, ask the AI to use `tidy()` with its standard names.

### Interpret it yourself first

In one or two sentences, say what the odds ratio for `exposure_status` means. Include the confidence interval. **Write your own sentence before asking the AI for one**, then compare the two. To ask the AI, give it only the odds ratio and its confidence interval, not the whole table.

One caution: an odds ratio is not "times as likely". When the outcome is common, as it is here, the odds ratio is much larger than the ratio of the two probabilities.

**My interpretation:** *(double-click to edit)*

(Remember that these data are simulated. The result illustrates the method; it says nothing about real medications.)

### Catch the mistake

Here is a request and an AI's answer.

**The request:** "Fit a logistic regression of the condition on drug exposure, adjusting for age and sex, and tell me the odds ratio for drug exposure."

**The AI's answer:** "The odds ratio for drug exposure is 2.16, so exposed participants have about twice the odds of the condition."

In [ ]:
model <- glm(condition_status ~ exposure_status + age + sex_at_birth,
             data = study_df, family = "binomial")
coef(model)["exposure_status"]

**What is wrong?** Use the three things to know at the top of this section. Then calculate the correct value in the cell below.

In [ ]:
# The correct odds ratio

### Ask about assumptions

Every statistical model rests on assumptions about the data. Ask the AI: *"What assumptions does this logistic regression make, and how do I check them in R?"* Regression results should not be used to draw conclusions until those checks have been done. See also the [Additional Resources](https://icohr-lab.github.io/aou-workshop/resources.html) page.

## 6. Taking this to the Researcher Workbench

The way of working is the same with real *All of Us* data, with one firm difference: **code goes in; data never comes out.** Write your prompt and get code outside the Workbench, then copy the code into your Workbench notebook and run it there.

Here again is the table from notebook B1.

| Safe to give an AI tool | Never give an AI tool |
|---|---|
| Your code | Rows of data, even one |
| Column names and types (`describe_columns()`) | The output of `head()`, `str()`, or a printed dataframe |
| The names of the categories in a column | `person_id` values, dates, or anything else about one person |
| Error messages, once you have read them and removed any data values they quote | Screenshots or plots that show individual participants |
| Your research question and analysis plan | Tables or counts in which any cell is below 20 |
| Summary statistics with no participant count below 20 | |

See the [*All of Us* policy on AI tools](https://support.researchallofus.org/hc/en-us/articles/34814131370388). When in doubt, ask before you paste.

`describe_columns()` is not built into R. To use it in the Workbench, copy its three lines from this notebook's setup cell into your own notebook.

---
## Recap

| Habit | Why |
|---|---|
| Choose the method yourself, before prompting | The AI will run whatever you ask for |
| Tell the AI each variable's type and values | It needs them to write the right test |
| Ask about anything in the output you did not request | Defaults are choices too |
| Exponentiate logistic regression coefficients | They are log odds ratios |
| Write your own interpretation first | Then compare it with the AI's |
| Ask what assumptions the method makes | A result is only as good as its assumptions |
| Code goes in; data never comes out | The rule for real *All of Us* data |

---
## Solutions

Try each exercise before looking. There is often more than one correct way to write it.

**Section 4: three more questions**

```r
# (a) One-way ANOVA
summary(aov(SBP ~ sex_at_birth, data = study_df))

# (b) Chi-square test
chisq.test(table(study_df$exposure_status, study_df$condition_status))

# (c) Linear regression
tidy(lm(SBP ~ age, data = study_df))
```

**Section 5: logistic regression**

`family = "binomial"` makes it a logistic regression. `exponentiate = TRUE` converts the coefficients to odds ratios.

```r
logistic_model <- glm(
  condition_status ~ exposure_status + age + sex_at_birth,
  data = study_df,
  family = "binomial"
)
tidy_results <- tidy(logistic_model, exponentiate = TRUE, conf.int = TRUE)
tidy_results
```

**Catch the mistake**

2.16 is the model's coefficient, which is a **log** odds ratio. The odds ratio is its exponential: 8.64, not 2.16. Reporting a logistic regression coefficient as an odds ratio is a common and serious error, and an AI can make it as easily as a person can.

```r
exp(coef(model)["exposure_status"])
```

**A sample interpretation of the odds ratio**

After adjusting for age and sex, people exposed to the drug had 8.6 times the odds of the condition compared with unexposed people (95% confidence interval 3.3 to 24.3).

---
## Next

You have finished Path B. You can now use an AI assistant to clean, combine, explore, and analyze a dataset in R, and you know how to check its work.

The next step is to do the same with real data in the *All of Us* Researcher Workbench. See the [workshop website](https://icohr-lab.github.io/aou-workshop/) for how to get started there.